# Lecture 6: Save a Ridge Model for Deployment

## Algerian Forest Fires project

We trained several regression models. This lesson chooses Ridge, then saves the model and its StandardScaler so a future app can predict without retraining.

**Road map:** train Ridge → save preprocessing + model → reload them → predict one new example → learn the safer pipeline option.

## Why save two objects?

A Ridge model was trained on **scaled** inputs. At deployment time, new inputs must receive the **same scaling** before reaching the model.

Diagram: New fire-weather values → saved StandardScaler → saved Ridge model → predicted FWI

If we save only the Ridge model, predictions can be wrong because the input scale is different.

In [ ]:
# Cell 1: imports and cleaned project data
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('Model Training Practicals/Algerian_forest_fires_cleaned_dataset.csv')
df.head()

### What changed here?

pickle writes Python objects to files and reads them back later. Path makes file locations clear. We use the same cleaned CSV as the earlier training lessons so this notebook is fully runnable.

In [ ]:
# Cell 2: prepare the exact input columns used by the model
df['Classes'] = (df['Classes'].astype(str).str.strip().str.lower()
                 .map({'not fire': 0, 'fire': 1}))
target = 'FWI'
feature_names = [column for column in df.columns if column not in ['day', 'month', 'year', target]]
X = df[feature_names]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Features in model order:', feature_names)

### Why save feature names?

A deployed app must supply the same columns in the same order as training. Storing feature_names removes guesswork and helps catch missing inputs.

In [ ]:
# Cell 3: fit the scaler ONLY on training data, then train Ridge
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
test_prediction = ridge_model.predict(X_test_scaled)
print('Test MAE:', round(mean_absolute_error(y_test, test_prediction), 3))
print('Test R2:', round(r2_score(y_test, test_prediction), 3))

## Save the two-file deployment package

The next cell follows the transcript: one file for preprocessing and one for the model. We also save the feature list, because input order matters.

Security note: only load pickle files you trust. A malicious pickle can run harmful code while loading.

In [ ]:
# Cell 4: write the scaler, Ridge model, and feature names to disk
artifact_dir = Path('Model Training Practicals/model_artifacts')
artifact_dir.mkdir(exist_ok=True)
with open(artifact_dir / 'scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)
with open(artifact_dir / 'ridge_model.pkl', 'wb') as file:
    pickle.dump(ridge_model, file)
with open(artifact_dir / 'feature_names.pkl', 'wb') as file:
    pickle.dump(feature_names, file)
for path in sorted(artifact_dir.iterdir()):
    print(f'{path.name}: {path.stat().st_size} bytes')

### Read the file-writing code

- wb means write binary mode.
- pickle.dump puts one Python object into the open file.
- The with-open pattern closes each file automatically.
- Do not retrain the scaler in the app. Load this fitted scaler instead.

In [ ]:
# Cell 5: reload the saved objects as a deployment app would
with open(artifact_dir / 'scaler.pkl', 'rb') as file:
    loaded_scaler = pickle.load(file)
with open(artifact_dir / 'ridge_model.pkl', 'rb') as file:
    loaded_ridge = pickle.load(file)
with open(artifact_dir / 'feature_names.pkl', 'rb') as file:
    loaded_features = pickle.load(file)
print('Loaded feature order:', loaded_features)
print('Loaded Ridge alpha:', loaded_ridge.alpha)

In [ ]:
# Cell 6: predict one new case by using the loaded scaler and model
new_case = X_test.iloc[[0]][loaded_features].copy()
new_case_scaled = loaded_scaler.transform(new_case)
new_prediction = loaded_ridge.predict(new_case_scaled)[0]
actual_fwi = y_test.iloc[0]
print('New input values:')
display(new_case)
print(f'Predicted FWI: {new_prediction:.2f}')
print(f'Actual FWI (held-out example): {actual_fwi:.2f}')

In [ ]:
# Cell 7: visual check that saved and original models agree
reloaded_prediction = loaded_ridge.predict(loaded_scaler.transform(X_test[loaded_features]))
plt.figure(figsize=(6, 4))
plt.scatter(test_prediction, reloaded_prediction, color='#4C78A8', alpha=0.8)
limits = [min(test_prediction.min(), reloaded_prediction.min()), max(test_prediction.max(), reloaded_prediction.max())]
plt.plot(limits, limits, '--', color='crimson', label='Same prediction line')
plt.xlabel('Original Ridge prediction')
plt.ylabel('Reloaded Ridge prediction')
plt.title('Saved artifacts reproduce the model output')
plt.legend()
plt.show()
print('Predictions identical:', np.allclose(test_prediction, reloaded_prediction))

## Better option: save one pipeline

The two-file approach is useful for learning. In production, one pipeline is often safer because scaling and prediction cannot be accidentally separated. The pipeline still stores the fitted scaler and the fitted Ridge model.

In [ ]:
# Cell 8: train, save, and reload one complete pipeline
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_pipeline.fit(X_train, y_train)
pipeline_path = artifact_dir / 'ridge_pipeline.pkl'
with open(pipeline_path, 'wb') as file:
    pickle.dump(ridge_pipeline, file)
with open(pipeline_path, 'rb') as file:
    loaded_pipeline = pickle.load(file)
pipeline_prediction = loaded_pipeline.predict(new_case)[0]
print(f'One-pipeline prediction: {pipeline_prediction:.2f}')
print('Matches two-file prediction:', np.isclose(new_prediction, pipeline_prediction))

## Quick revision card

1. Choose a model after evaluating it on a held-out test set.
2. Save the fitted preprocessing object as well as the fitted model.
3. Save feature names and use them in the same order at prediction time.
4. wb saves a pickle; rb loads one.
5. Reloaded artifacts should reproduce the original model predictions.
6. Only load pickle files from trusted sources.

**One-line interview answer:** Deployment uses the saved fitted scaler and saved model so that new data receives exactly the same preprocessing as training data.